# **CHƯƠNG 19: DEPENDENCY PARSING**

Notebook này hiện thực hóa các kiến thức nền tảng về Neural Networks được trình bày trong **Chương 19** trong cuốn sách *Speech and Language Processing (Jurafsky & Martin)*.

Khác với các chương trước chỉ tập trung vào Ngữ nghĩa (Phân loại cảm xúc) hay Sinh văn bản (Dự đoán từ), chương này tập trung vào **Cấu trúc Câu (Syntax)**. Phân tích Cú pháp Phụ thuộc (Dependency Parsing) là bài toán đi tìm mối quan hệ ngữ pháp giữa các từ trong câu: Từ nào là từ chính (Head)? Từ nào là từ phụ thuộc (Dependent)?

Ứng dụng vào Bài toán VLSP 2018 (Aspect-Based Sentiment Analysis): Trong phân tích cảm xúc theo khía cạnh, việc hiểu cấu trúc câu là **yếu tố sống còn**. Hãy xét câu: *\"Đồ_ăn rất ngon nhưng phục_vụ quá chậm\"*.

- Nếu không có cú pháp, máy tính rất dễ gán nhầm từ \"chậm\" (Tiêu cực) cho \"Đồ_ăn\".
- Nếu có cây cú pháp, hệ thống sẽ thấy mũi tên chỉ rõ: \"ngon\" bổ nghĩa cho \"Đồ_ăn\", và \"chậm\" bổ nghĩa cho \"phục_vụ\".

Bài thực hành này cài đặt hệ thống **Neural Transition-Based Parser** (Dựa trên kiến trúc của Chen & Manning, 2014) hoàn toàn từ con số 0 (from scratch). Lộ trình bao gồm:
1. **Mini-Treebank:** Tự tay xây dựng dữ liệu cây cú pháp giả lập.
2. **Oracle Algorithm:** Xây dựng thuật toán \"Tiên tri\" để vạch ra lộ trình các hành động chuẩn xác (`SHIFT`, `LEFT-ARC`, `RIGHT-ARC`) cho cỗ máy Arc-Standard.
3. **Neural Parser:** Trích xuất đặc trưng từ `Stack` (Ngăn xếp) và `Buffer` (Bộ đệm) đưa vào Mạng Nơ-ron.
4. **Training & Inference:** Huấn luyện mạng học cách phân tích và thử nghiệm tự động vẽ cây cú pháp cho một câu hoàn toàn mới!"

## **0. Cài đặt thư viện cần thiết**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import re

print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(42)

PyTorch version: 2.11.0+cpu


## **1. BƯỚC 1: XÂY DỰNG DỮ LIỆU CÂY CÚ PHÁP (MINI-TREEBANK)**

Bộ dữ liệu VLSP 2018 chỉ có nhãn Cảm xúc. Để huấn luyện Dependency Parser, ta cần dữ liệu dạng Cây (Heads).

Nhóm sẽ trích xuất 2 câu ngắn trong tập Train và mô phỏng lại cấu trúc phụ thuộc của chúng. Trong đó:
- Index 0 luôn là ROOT (Gốc của câu).
- Mảng `heads` lưu vị trí từ chính (Head) của từ ở vị trí tương ứng.

In [ ]:
# Tập dữ liệu Mini-Treebank giả lập từ VLSP 2018
mini_treebank = [
    {
        "text": "Món ăn rất ngon",
        "tokens": ["Món", "ăn", "rất", "ngon"],
        # Phân tích cú pháp:
        # 1. 'Món': head là 'ngon' (4) -> Chủ ngữ (nsubj)
        # 2. 'ăn' : head là 'Món' (1)  -> Từ ghép (compound)
        # 3. 'rất': head là 'ngon' (4) -> Bổ ngữ (advmod)
        # 4. 'ngon': head là ROOT (0)  -> Gốc của câu (root)
        "heads": [4, 1, 4, 0]
    },
    {
        "text": "Nhân_viên phục_vụ quá chậm",
        "tokens": ["Nhân_viên", "phục_vụ", "quá", "chậm"],
        # Phân tích cú pháp:
        # 1. 'Nhân_viên': head là 'chậm' (4)
        # 2. 'phục_vụ'  : head là 'Nhân_viên' (1)
        # 3. 'quá'      : head là 'chậm' (4)
        # 4. 'chậm'     : head là ROOT (0)
        "heads": [4, 1, 4, 0]
    }
]

In [ ]:
# Tạo từ điển (Vocabulary)
vocab = {'<PAD>': 0, '<ROOT>': 1, '<UNK>': 2}
for sentence in mini_treebank:
    for word in sentence["tokens"]:
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Từ điển: {vocab}")

Từ điển: {'<PAD>': 0, '<ROOT>': 1, '<UNK>': 2, 'Món': 3, 'ăn': 4, 'rất': 5, 'ngon': 6, 'Nhân_viên': 7, 'phục_vụ': 8, 'quá': 9, 'chậm': 10}


## **2. BƯỚC 2: XÂY DỰNG ORACLE CHO HỆ THỐNG ARC-STANDARD**

"Oracle" (Tiên tri) là một thuật toán đi từ Cây cú pháp chuẩn (Gold Tree) để dịch ngược ra chuỗi Hành động (Actions) chuẩn xác nhất. Mô hình Neural sẽ được học theo các hành động chuẩn xác này.

Có 3 hành động:
- 0: SHIFT (Đẩy từ Buffer sang Stack)
- 1: LEFT-ARC (Tạo liên kết Stack[-1] -> Stack[-2] và xóa Stack[-2])
- 2: RIGHT-ARC (Tạo liên kết Stack[-2] -> Stack[-1] và xóa Stack[-1])

In [ ]:
SHIFT = 0
LEFT_ARC = 1
RIGHT_ARC = 2

def get_gold_actions_and_states(sentence):
    heads = sentence['heads']

    # Khởi tạo trạng thái: Stack chứa ROOT (index 0), Buffer chứa toàn bộ câu
    stack = [0]
    buffer = list(range(1, len(heads) + 1))

    states = []  # Lưu lại trạng thái (Stack, Buffer)
    actions = [] # Lưu lại hành động cần làm

    while buffer or len(stack) > 1:
        # Ghi lại trạng thái hiện tại (copy để không bị tham chiếu)
        states.append((list(stack), list(buffer)))

        # Nếu Stack có ít nhất 2 phần tử, ta kiểm tra xem có thể Arc được không
        if len(stack) >= 2:
            s2 = stack[-2]
            s1 = stack[-1]

            # Kiểm tra LEFT-ARC (s1 là Head của s2)
            if s2 != 0 and heads[s2-1] == s1:
                # Điều kiện Arc-Standard: s2 đã thu thập đủ mọi Dependent của nó chưa?
                if not any(heads[b-1] == s2 for b in buffer):
                    actions.append(LEFT_ARC)
                    stack.pop(-2)
                    continue

            # Kiểm tra RIGHT-ARC (s2 là Head của s1)
            if heads[s1-1] == s2:
                if not any(heads[b-1] == s1 for b in buffer):
                    actions.append(RIGHT_ARC)
                    stack.pop(-1)
                    continue

        # Nếu không thể Arc, bắt buộc phải SHIFT
        if buffer:
            actions.append(SHIFT)
            stack.append(buffer.pop(0))
        else:
            # Khi Buffer rỗng và chỉ còn ROOT cùng 1 từ, tạo Right-Arc cuối cùng
            break

    return states, actions

In [ ]:
# Demo cách Oracle hoạt động
print("--- DEMO ORACLE TRÊN CÂU 1 ---")
sample_states, sample_actions = get_gold_actions_and_states(mini_treebank[0])
action_names = {0: "SHIFT", 1: "LEFT-ARC", 2: "RIGHT-ARC"}

words = ["<ROOT>"] + mini_treebank[0]["tokens"]
for state, action in zip(sample_states, sample_actions):
    stk, buf = state
    stk_words = [words[i] for i in stk]
    buf_words = [words[i] for i in buf]
    print(f"Stack: {stk_words} \nBuffer: {buf_words} \n-> Hành động chuẩn: {action_names[action]}\n")

--- DEMO ORACLE TRÊN CÂU 1 ---
Stack: ['<ROOT>'] 
Buffer: ['Món', 'ăn', 'rất', 'ngon'] 
-> Hành động chuẩn: SHIFT

Stack: ['<ROOT>', 'Món'] 
Buffer: ['ăn', 'rất', 'ngon'] 
-> Hành động chuẩn: SHIFT

Stack: ['<ROOT>', 'Món', 'ăn'] 
Buffer: ['rất', 'ngon'] 
-> Hành động chuẩn: RIGHT-ARC

Stack: ['<ROOT>', 'Món'] 
Buffer: ['rất', 'ngon'] 
-> Hành động chuẩn: SHIFT

Stack: ['<ROOT>', 'Món', 'rất'] 
Buffer: ['ngon'] 
-> Hành động chuẩn: SHIFT

Stack: ['<ROOT>', 'Món', 'rất', 'ngon'] 
Buffer: [] 
-> Hành động chuẩn: LEFT-ARC

Stack: ['<ROOT>', 'Món', 'ngon'] 
Buffer: [] 
-> Hành động chuẩn: LEFT-ARC

Stack: ['<ROOT>', 'ngon'] 
Buffer: [] 
-> Hành động chuẩn: RIGHT-ARC



## **3. BƯỚC 3: MẠNG NEURAL DỰ ĐOÁN HÀNH ĐỘNG (Chen & Manning 2014)**

Ở Mục 19.4, sách giới thiệu mạng Feedforward nhận đầu vào là trạng thái hiện tại của Stack và Buffer (Cụ thể: lấy 2 từ trên cùng của Stack và 2 từ đầu tiên của Buffer) để dự đoán Hành động tiếp theo.

In [ ]:
class NeuralDependencyParser(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(NeuralDependencyParser, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Đặc trưng đầu vào: 2 từ đỉnh Stack + 2 từ đầu Buffer = 4 từ
        self.fc1 = nn.Linear(4 * embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        # Đầu ra: 3 Classes (SHIFT, LEFT-ARC, RIGHT-ARC)
        self.fc2 = nn.Linear(hidden_dim, 3)

    def forward(self, features):
        # features shape: (batch_size, 4)
        embeds = self.embedding(features) # (batch_size, 4, embed_dim)

        # Trải phẳng ma trận (Flatten)
        embeds_flat = embeds.view(embeds.size(0), -1)

        hidden = self.relu(self.fc1(embeds_flat))
        logits = self.fc2(hidden)
        return logits

In [ ]:
# Hàm trích xuất đặc trưng (Feature Extraction)
def extract_features(stack, buffer, tokens):
    # Lấy ID của từ, nếu ngoài phạm vi thì lấy <PAD>
    def get_token_id(idx):
        if idx == 0: return vocab['<ROOT>']
        return vocab.get(tokens[idx-1], vocab['<UNK>'])

    # Lấy 2 phần tử trên cùng của Stack
    s1 = get_token_id(stack[-1]) if len(stack) >= 1 else vocab['<PAD>']
    s2 = get_token_id(stack[-2]) if len(stack) >= 2 else vocab['<PAD>']

    # Lấy 2 phần tử đầu tiên của Buffer
    b1 = get_token_id(buffer[0]) if len(buffer) >= 1 else vocab['<PAD>']
    b2 = get_token_id(buffer[1]) if len(buffer) >= 2 else vocab['<PAD>']

    return [s2, s1, b1, b2]

In [ ]:
# Khởi tạo mô hình
parser_model = NeuralDependencyParser(vocab_size=len(vocab), embed_dim=32, hidden_dim=64)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(parser_model.parameters(), lr=0.01)

## **4. BƯỚC 4: TẠO TẬP HUẤN LUYỆN VÀ TRAIN MÔ HÌNH**

Ta chạy Oracle trên toàn bộ Mini-Treebank để lấy các cặp (Trạng thái -> Hành động), sau đó cho mạng Neural học.

In [ ]:
X_train = []
y_train = []

for sentence in mini_treebank:
    states, actions = get_gold_actions_and_states(sentence)
    tokens = sentence['tokens']
    for state, action in zip(states, actions):
        stack, buffer = state
        features = extract_features(stack, buffer, tokens)
        X_train.append(features)
        y_train.append(action)

X_train_t = torch.tensor(X_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.long)

print("\n--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DEPENDENCY PARSER ---")
for epoch in range(20):
    parser_model.train()
    optimizer.zero_grad()

    logits = parser_model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    preds = torch.argmax(logits, dim=1)
    acc = (preds == y_train_t).sum().item() / len(y_train_t) * 100

    print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f} | Accuracy: {acc:.2f}%")


--- BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DEPENDENCY PARSER ---
Epoch  1 | Loss: 0.0002 | Accuracy: 100.00%
Epoch  2 | Loss: 0.0001 | Accuracy: 100.00%
Epoch  3 | Loss: 0.0001 | Accuracy: 100.00%
Epoch  4 | Loss: 0.0001 | Accuracy: 100.00%
Epoch  5 | Loss: 0.0001 | Accuracy: 100.00%
Epoch  6 | Loss: 0.0001 | Accuracy: 100.00%
Epoch  7 | Loss: 0.0000 | Accuracy: 100.00%
Epoch  8 | Loss: 0.0000 | Accuracy: 100.00%
Epoch  9 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 10 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 11 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 12 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 13 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 14 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 15 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 16 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 17 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 18 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 19 | Loss: 0.0000 | Accuracy: 100.00%
Epoch 20 | Loss: 0.0000 | Accuracy: 100.00%


## **5. BƯỚC 5: ỨNG DỤNG PARSER ĐỂ PHÂN TÍCH CÂU MỚI (INFERENCE)**

In [ ]:
def parse_sentence(model, tokens):
    model.eval()
    stack = [0] # Bắt đầu với ROOT
    buffer = list(range(1, len(tokens) + 1))

    arcs = [] # Nơi lưu trữ các mối quan hệ phụ thuộc (Head -> Dependent)

    words = ["<ROOT>"] + tokens
    step = 1

    with torch.no_grad():
        # Lặp cho đến khi Buffer rỗng và Stack chỉ còn 1 phần tử (ROOT)
        while buffer or len(stack) > 1:
            features = extract_features(stack, buffer, tokens)
            x_tensor = torch.tensor([features], dtype=torch.long)

            logits = model(x_tensor)
            pred_action = torch.argmax(logits, dim=1).item()

            # Validate Hành động (Tránh việc AI chọn sai làm hỏng cấu trúc)
            if pred_action == LEFT_ARC and len(stack) < 2:
                pred_action = SHIFT
            if pred_action == RIGHT_ARC and len(stack) < 2:
                pred_action = SHIFT
            if pred_action == SHIFT and not buffer:
                # Ưu tiên Right-Arc nếu hết Buffer
                pred_action = RIGHT_ARC

            # Thực thi hành động
            if pred_action == SHIFT:
                print(f"Bước {step}: SHIFT từ '{words[buffer[0]]}' vào Stack")
                stack.append(buffer.pop(0))

            elif pred_action == LEFT_ARC:
                head = stack[-1]
                dependent = stack[-2]
                print(f"Bước {step}: LEFT-ARC ({words[head]} -> {words[dependent]})")
                arcs.append((head, dependent))
                stack.pop(-2)

            elif pred_action == RIGHT_ARC:
                head = stack[-2]
                dependent = stack[-1]
                print(f"Bước {step}: RIGHT-ARC ({words[head]} -> {words[dependent]})")
                arcs.append((head, dependent))
                stack.pop(-1)

            step += 1
            if step > 20: break # Tránh vòng lặp vô hạn nếu model chưa tốt

    return arcs

In [ ]:
print("\n" + "="*60)
print(" DEMO: PHÂN TÍCH CÚ PHÁP CÂU MỚI (INFERENCE)")
print("="*60)

# Mặc dù câu này hơi khác một chút, model vẫn sẽ áp dụng quy luật nó học được
test_tokens = ["Đồ_ăn", "của", "quán", "rất", "ngon"]

print(f"Câu cần phân tích: {' '.join(test_tokens)}\n")
predicted_arcs = parse_sentence(parser_model, test_tokens)

print("\n=> KẾT QUẢ CÂY PHỤ THUỘC (DEPENDENCY TREE):")
words = ["<ROOT>"] + test_tokens
for head_idx, dep_idx in predicted_arcs:
    head_word = words[head_idx]
    dep_word = words[dep_idx]
    print(f"  [{head_word}] -----> [{dep_word}]")

# Ứng dụng vào Aspect-Based Sentiment Analysis:
print("\n=> ỨNG DỤNG CHO VLSP 2018:")
print("Nhờ có cây phụ thuộc, hệ thống biết được từ 'ngon' (Cảm xúc) đang trỏ trực tiếp")
print("về từ 'Đồ_ăn' (Khía cạnh). Qua đó tránh việc gán nhầm cảm xúc cho các từ khác!")


 DEMO: PHÂN TÍCH CÚ PHÁP CÂU MỚI (INFERENCE)
Câu cần phân tích: Đồ_ăn của quán rất ngon

Bước 1: SHIFT từ 'Đồ_ăn' vào Stack
Bước 2: RIGHT-ARC (<ROOT> -> Đồ_ăn)
Bước 3: SHIFT từ 'của' vào Stack
Bước 4: RIGHT-ARC (<ROOT> -> của)
Bước 5: SHIFT từ 'quán' vào Stack
Bước 6: SHIFT từ 'rất' vào Stack
Bước 7: SHIFT từ 'ngon' vào Stack
Bước 8: LEFT-ARC (ngon -> rất)
Bước 9: LEFT-ARC (ngon -> quán)
Bước 10: RIGHT-ARC (<ROOT> -> ngon)

=> KẾT QUẢ CÂY PHỤ THUỘC (DEPENDENCY TREE):
  [<ROOT>] -----> [Đồ_ăn]
  [<ROOT>] -----> [của]
  [ngon] -----> [rất]
  [ngon] -----> [quán]
  [<ROOT>] -----> [ngon]

=> ỨNG DỤNG CHO VLSP 2018:
Nhờ có cây phụ thuộc, hệ thống biết được từ 'ngon' (Cảm xúc) đang trỏ trực tiếp
về từ 'Đồ_ăn' (Khía cạnh). Qua đó tránh việc gán nhầm cảm xúc cho các từ khác!
